In [ ]:
from pathlib import Path

DATA_DIR = Path(
    "/blue/bala1s/krishnap.kalivel/TLFHydrodynamics/"
    "thinfilm-closure-ml/datasets/BasC/param_space/ClosureData"
)

RE = 40
CA = 0.010
BETA = 90.0  # Degrees

# Display settings
X_LIMITS = None       # Example: (400, 600)
TIME_LIMITS = None    # Example: (10, 30)
H_LIMITS = None       # Example: (0.5, 2.5)

FRAME_STRIDE = 1      # Animate every stored snapshot; increase to skip frames.
FPS = 15             # Playback speed

ca_tag = f"{CA:.3f}".replace(".", "_")
beta_tag = f"{BETA:g}".replace(".", "_")

FILE_PATH = DATA_DIR / (
    f"thinfilm_closure_data_Re{RE:03d}_"
    f"Ca{ca_tag}_theta{beta_tag}.h5"
)

print(FILE_PATH)

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

if not FILE_PATH.is_file():
    raise FileNotFoundError(FILE_PATH)

if FRAME_STRIDE < 1 or int(FRAME_STRIDE) != FRAME_STRIDE:
    raise ValueError("FRAME_STRIDE must be a positive integer.")
if FPS <= 0:
    raise ValueError("FPS must be positive.")

with h5py.File(FILE_PATH, "r") as f:
    x = np.asarray(f["x"][:], dtype=float).reshape(-1)
    t = np.asarray(f["t"][:], dtype=float).reshape(-1)
    h_raw = np.asarray(f["h"][:], dtype=float)

nx, nt = len(x), len(t)

if nx < 2 or nt < 2:
    raise ValueError("At least two spatial points and two snapshots are needed.")

if not np.all(np.isfinite(x)) or not np.all(np.diff(x) > 0):
    raise ValueError("x must be finite and strictly increasing.")

if not np.all(np.isfinite(t)) or not np.all(np.diff(t) > 0):
    raise ValueError("t must be finite and strictly increasing.")

# Standardize the layout to h[x_index, time_index].
if nx == nt and h_raw.shape == (nx, nt):
    raise ValueError("Square h array: specify its spatial axis explicitly.")
elif h_raw.shape == (nx, nt):
    h = h_raw
elif h_raw.shape == (nt, nx):
    h = h_raw.T
else:
    raise ValueError(
        f"Unexpected h shape {h_raw.shape}; expected {(nx, nt)} "
        f"or {(nt, nx)}."
    )

del h_raw


def range_mask(values, limits, name):
    if limits is None:
        return np.ones(values.size, dtype=bool)

    lower, upper = limits
    if lower >= upper:
        raise ValueError(f"{name}: lower limit must be below upper limit.")

    return (values >= lower) & (values <= upper)


x_mask = range_mask(x, X_LIMITS, "X_LIMITS")
time_mask = range_mask(t, TIME_LIMITS, "TIME_LIMITS")

frame_indices = np.flatnonzero(time_mask)[::int(FRAME_STRIDE)]

if x_mask.sum() < 2 or frame_indices.size < 2:
    raise ValueError("Selected ranges must retain >= 2 x points and snapshots.")

x_plot = x[x_mask]
h_plot = h[np.ix_(np.flatnonzero(x_mask), frame_indices)]

finite_values = h_plot[np.isfinite(h_plot)]
if finite_values.size == 0:
    raise ValueError("Selected profiles contain no finite heights.")

if H_LIMITS is None:
    lower, upper = finite_values.min(), finite_values.max()
    padding = 0.05 * max(upper - lower, 0.1)
    height_limits = (lower - padding, upper + padding)
else:
    if H_LIMITS[0] >= H_LIMITS[1]:
        raise ValueError("H_LIMITS must satisfy lower < upper.")
    height_limits = H_LIMITS

# Display nonfinite values as gaps.
h_plot = np.where(np.isfinite(h_plot), h_plot, np.nan)

fig, ax = plt.subplots(
    figsize=(12, 4.5),
    constrained_layout=True,
)

line, = ax.plot(
    x_plot, h_plot[:, 0],
    color="tab:blue",
    linewidth=1.5,
)

ax.axhline(
    1.0,
    color="gray",
    linestyle=":",
    linewidth=1.0,
    label=r"$h=1$",
)

ax.set_xlabel(r"$x$", fontsize=14)
ax.set_ylabel(r"$h(x,t)$", fontsize=14)
ax.set_xlim(x_plot[0], x_plot[-1])
ax.set_ylim(*height_limits)
ax.tick_params(labelsize=12)
ax.grid(alpha=0.3)
ax.legend()

title = ax.set_title("")


def update(frame):
    it = frame_indices[frame]

    line.set_ydata(h_plot[:, frame])
    title.set_text(
        rf"$Re={RE:g},\ Ca={CA:g},\ \beta={BETA:g}^\circ$"
        "\n"
        rf"$t={t[it]:.6g}$"
    )

    return line, title


update(0)

animation = FuncAnimation(
    fig,
    update,
    frames=len(frame_indices),
    interval=1000 / FPS,
    blit=False,
    repeat=True,
)

print(f"Loaded h: {h.shape} (nx, nt)")
print(f"Grid spacing: {x[1] - x[0]:.8g}")
print(f"Animation frames: {len(frame_indices)}")
print("Playback uses a fixed frame rate; titles show actual stored times.")

# Embedded controls work without ffmpeg.
with plt.rc_context({"animation.embed_limit": 100.0}):
    animation_html = animation.to_jshtml(
        fps=FPS,
        default_mode="loop",
    )

plt.close(fig)
display(HTML(animation_html))

In [ ]:
SELECTED_TIME = 580.0

if not np.isfinite(SELECTED_TIME):
    raise ValueError("SELECTED_TIME must be finite.")
if not t[0] <= SELECTED_TIME <= t[-1]:
    raise ValueError(
        f"SELECTED_TIME must lie within [{t[0]:g}, {t[-1]:g}]."
    )

it = int(np.argmin(np.abs(t - SELECTED_TIME)))
selected_time = float(t[it])

fig, ax = plt.subplots(figsize=(12, 4.5), constrained_layout=True)

ax.plot(
    x,
    h[:, it],
    color="black",
    linewidth=1.5,
)

ax.axhline(
    1.0,
    color="gray",
    linestyle=":",
    linewidth=1.0,
)

ax.set_xlabel(r"$x$", fontsize=16)
ax.set_ylabel(r"$h(x,t)$", fontsize=16)
ax.set_title(
    rf"$Re={RE:g},\ Ca={CA:g},\ \beta={BETA:g}^\circ$"
    "\n"
    rf"$t={selected_time:.6g}$",
    fontsize=16,
)

ax.set_xlim(*(X_LIMITS if X_LIMITS is not None else (x[0], x[-1])))

if H_LIMITS is not None:
    ax.set_ylim(*H_LIMITS)

ax.tick_params(axis="both", labelsize=14)
ax.grid(alpha=0.3)

print(f"Requested time: {SELECTED_TIME:g}")
print(f"Selected stored time: {selected_time:.8g} (index {it})")

plt.show()